In [3]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\sujan\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [2]:
pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 10.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/865.2 kB ? eta -:--:--
   ---------------------------------------- 865.2/865.2 kB 4.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/53.7 MB ? eta -:--:--
   - -------------------------------------- 2.4/53.7 MB 11.9 MB/s eta 0:00:05
   --- --------------------


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
DATASET_PATH = r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset"

In [7]:
model = YOLO("yolo11n-cls.pt")

In [8]:
results = model.train(
    data=DATASET_PATH,
    epochs=15,
    imgsz=224,
    batch=32,
    lr0=0.001,
    patience=5,
    workers=2,
    project="runs/classify",
    name="RiceDisease_YOLO11"
)

Ultralytics 8.4.135  Python-3.13.1 torch-2.13.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mos

In [9]:
from ultralytics import YOLO
from sklearn.metrics import classification_report
import numpy as np
from pathlib import Path
from PIL import Image

# Load best model
model = YOLO(r"C:\Users\sujan\Desktop\final project2\DATASET\runs\classify\runs\classify\RiceDisease_YOLO11-2\weights\best.pt")

test_dir = Path(r"C:\Users\sujan\Desktop\final project2\DATASET\Processed_RiceDisease_Dataset\test")

class_names = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])

y_true = []
y_pred = []

for class_idx, class_name in enumerate(class_names):
    class_folder = test_dir / class_name

    for img_path in class_folder.glob("*.*"):
        try:
            result = model.predict(
                source=str(img_path),
                verbose=False
            )[0]

            pred = int(result.probs.top1)

            y_true.append(class_idx)
            y_pred.append(pred)

        except:
            pass

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

                       precision    recall  f1-score   support

bacterial_leaf_blight       1.00      0.98      0.99        55
           brown_spot       0.87      0.98      0.92        59
              healthy       1.00      0.94      0.97        66
           leaf_blast       0.98      0.92      0.95        63
           leaf_scald       1.00      1.00      1.00        54
            leaf_smut       0.83      0.83      0.83         6
    narrow_brown_spot       0.98      1.00      0.99        62

             accuracy                           0.97       365
            macro avg       0.95      0.95      0.95       365
         weighted avg       0.97      0.97      0.97       365



In [10]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(10,8))

disp.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=45
)

plt.title("YOLOv11 Confusion Matrix")
plt.show()

<Figure size 1000x800 with 2 Axes>

In [11]:
cm = confusion_matrix(
    y_true,
    y_pred,
    normalize="true"
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(10,8))

disp.plot(
    ax=ax,
    cmap="Blues",
    xticks_rotation=45,
    values_format=".2f"
)

plt.title("YOLOv11 Normalized Confusion Matrix")
plt.show()

<Figure size 1000x800 with 2 Axes>

In [12]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(
    r"C:\Users\sujan\Desktop\final project2\DATASET\runs\classify\runs\classify\RiceDisease_YOLO11-2\results.png"
)

plt.figure(figsize=(12,8))
plt.imshow(img)
plt.axis("off")
plt.show()

<Figure size 1200x800 with 1 Axes>

In [13]:
img = Image.open(
    r"C:\Users\sujan\Desktop\final project2\DATASET\runs\classify\runs\classify\RiceDisease_YOLO11-2\confusion_matrix_normalized.png"
)

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")
plt.show()

<Figure size 800x800 with 1 Axes>

In [14]:
import pandas as pd

csv_path = r"C:\Users\sujan\Desktop\final project2\DATASET\runs\classify\runs\classify\RiceDisease_YOLO11-2\results.csv"

df = pd.read_csv(csv_path)

print(df.columns)
print(df.head())

Index(['epoch', 'time', 'train/loss', 'metrics/accuracy_top1',
       'metrics/accuracy_top5', 'val/loss', 'lr/pg0', 'lr/pg1', 'lr/pg2'],
      dtype='object')
   epoch     time  train/loss  metrics/accuracy_top1  metrics/accuracy_top5  \
0      1  149.654     1.35792                0.80939                      1   
1      2  295.926     0.56951                0.84254                      1   
2      3  441.075     0.47222                0.61878                      1   
3      4  586.634     0.45634                0.79558                      1   
4      5  732.731     0.35481                0.91436                      1   

   val/loss    lr/pg0    lr/pg1    lr/pg2  
0   0.50592  0.000298  0.000298  0.000298  
1   0.41213  0.000561  0.000561  0.000561  
2   1.41725  0.000784  0.000784  0.000784  
3   0.56857  0.000729  0.000729  0.000729  
4   0.25224  0.000669  0.000669  0.000669  


In [15]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(
    r"C:\Users\sujan\Desktop\final project2\DATASET\runs\classify\runs\classify\RiceDisease_YOLO11-2\results.png"
)

plt.figure(figsize=(14,8))
plt.imshow(img)
plt.axis("off")
plt.show()

<Figure size 1400x800 with 1 Axes>

In [16]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "CNN",
        "ResNet50",
        "DenseNet121",
        "EfficientNetB3",
        "YOLOv11"
    ],
    
    "Accuracy": [
        81.64,
        94.79,
        89.32,  
        94.52, 
        97.20
    ]
})

results

,Model,Accuracy
0,CNN,81.64
1,ResNet50,94.79
2,DenseNet121,89.32
3,EfficientNetB3,94.52
4,YOLOv11,97.20


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Model Results
results = pd.DataFrame({
    "Model": [
        "CNN",
        "ResNet50",
        "DenseNet121",
        "EfficientNetB3",
        "YOLOv11"
    ],
    
    "Accuracy": [
        81.64,
        94.79,
        89.32,
        94.52,
        97.20
    ]
})

# Create Bar Graph
plt.figure(figsize=(10, 6))

bars = plt.bar(results["Model"], results["Accuracy"])

plt.title("Comparison of Deep Learning Models for Paddy Leaf Disease Detection",
          fontsize=14, fontweight='bold')
plt.xlabel("Models", fontsize=12)
plt.ylabel("Test Accuracy (%)", fontsize=12)

plt.ylim(75, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Display accuracy values on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height + 0.3,
             f"{height:.2f}%",
             ha='center',
             fontsize=10,
             fontweight='bold')

# Save graph
plt.savefig("Model_Comparison_Bar_Graph.png", dpi=300, bbox_inches="tight")

plt.show()

print("✅ Bar graph saved as 'Model_Comparison_Bar_Graph.png'")

<Figure size 1000x600 with 1 Axes>

✅ Bar graph saved as 'Model_Comparison_Bar_Graph.png'


In [1]:
import pandas as pd

results_path = r"C:\Users\sujan\Desktop\final project2\DATASET\runs\classify\runs\classify\RiceDisease_YOLO11-2\results.csv"

results = pd.read_csv(results_path)

print(results)

    epoch      time  train/loss  metrics/accuracy_top1  metrics/accuracy_top5  \
0       1   149.654     1.35792                0.80939                      1   
1       2   295.926     0.56951                0.84254                      1   
2       3   441.075     0.47222                0.61878                      1   
3       4   586.634     0.45634                0.79558                      1   
4       5   732.731     0.35481                0.91436                      1   
5       6   878.027     0.33650                0.91713                      1   
6       7  1022.290     0.28155                0.91713                      1   
7       8  1167.660     0.21841                0.93923                      1   
8       9  1315.840     0.21029                0.95028                      1   
9      10  1464.190     0.18679                0.95580                      1   
10     11  1611.050     0.15981                0.97238                      1   
11     12  1757.980     0.14